# Seven States That Break Q-Learning

**Topic:** The deadly triad &nbsp;·&nbsp; **Difficulty:** Hard

Tabular Q-learning converges. Add function approximation, bootstrapping, and
off-policy sampling at the same time and it can diverge, not oscillate but diverge,
with the weights growing without bound on a seven-state problem where the true
values are all zero.

Baird's counterexample is the standard demonstration. Run it, watch the weights
blow up, then add DQN's frozen target network and measure what it actually buys,
which is a large slowdown, not a cure.

## The data (created in the next cell)

- 7 states, 8 features. States 0-5 have `phi[i, i] = 2` and `phi[i, 7] = 1`,
  and state 6 has `phi[6, 6] = 1` and `phi[6, 7] = 2`.
- All rewards are `0`, so the true value function is `0` everywhere and is exactly
  representable (`w = 0`).
- The target policy always moves to state 6, and the behaviour policy samples
  states uniformly. `gamma = 0.99`, `lr = 0.01`, 1000 updates.
- `w0 = [1, 1, 1, 1, 1, 1, 10, 1]`.

## Your task

With **NumPy only**:

1. Semi-gradient TD update: sample a state `s` uniformly from all 7, the next
   state is always 6, and the TD error is
   `0 + gamma * (phi[6] . w_boot) - phi[s] . w`, with
   `w += lr * td * phi[s]`. The checks score against one reference run, so use
   a fresh `np.random.default_rng(11)` in each run and draw one state per update
   with `r.integers(S)`.
2. `norm_plain`: `||w||` after each of 1000 updates, with `w_boot = w`.
3. `norm_target`: the same run, but `w_boot` comes from a copy frozen every 50
   updates. Start it as `wt = w0.copy()` and refresh it after updates 50, 100,
   150, ..., which is when `(t + 1) % 50 == 0` for an update counter `t` that
   starts at 0.
4. Set `answers` to a dict with `"norm_plain"`, `"norm_target"` (length-1000
   each), `"final_plain"`, `"final_target"`, and `"growth_plain"`, the final
   norm divided by the initial one.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the plain run grows by more than 10x when the true answer is `w = 0`, and
that the frozen target more than halves the final norm.

In [ ]:
import numpy as np

S, K = 7, 8
Phi = np.zeros((S, K))
for i in range(6):
    Phi[i, i] = 2.0
    Phi[i, 7] = 1.0
Phi[6, 6] = 1.0
Phi[6, 7] = 2.0

gamma, lr, n_steps, sync = 0.99, 0.01, 1000, 50
w0 = np.array([1., 1., 1., 1., 1., 1., 10., 1.])
print(f"{S} states, {K} features. Every reward is 0, so w = 0 is the exact answer")
print(f"starting ||w|| = {np.linalg.norm(w0):.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("norm_plain", [1000], 141929.2944512743, -2251.0867659758933, 1e-06),
    ("norm_target", [1000], 43140.73793733849, -508.13092365032185, 1e-06),
    ("final_plain", [], 446.57245729807335, -74.77320572082209, 1e-06),
    ("final_target", [], 87.78977382079997, -14.699345449568636, 1e-06),
    ("growth_plain", [], 43.17178894728339, -7.2285986373236675, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "growth_plain" in answers:
    _div = answers["growth_plain"] > 10
    _slower = answers["final_target"] * 2 < answers["final_plain"]
    print(f"  {'plain run diverges':<24} {'ok' if _div else 'wrong'}  "
          f"(||w|| grows {answers['growth_plain']:.1f}x toward the answer w = 0)")
    print(f"  {'frozen target slows it':<24} {'ok' if _slower else 'wrong'}  "
          f"({answers['final_target']:.1f} vs {answers['final_plain']:.1f})")
    _ok = _ok and _div and _slower

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The features overlap, since every state shares component 7, so an update
aimed at one state moves all of them, and off-policy sampling means the states
you update are not the ones the bootstrap depends on. That combination (function approximation +
bootstrapping + off-policy) is Sutton and Barto's deadly triad, and Baird's
construction makes it diverge geometrically. Freezing the bootstrap for 50 steps
breaks the feedback within each block and slows the growth by several times, which
is why DQN needs it, but the norm still climbs, which is why DQN also needs
replay, reward clipping, and a small learning rate. No single stabilizer fixes the
triad.

```python
import numpy as np

def run(use_target):
    w = w0.copy()
    wt = w.copy()
    r = np.random.default_rng(11)
    norms = np.empty(n_steps)
    for t in range(n_steps):
        s = int(r.integers(S))
        boot = float(Phi[6] @ (wt if use_target else w))
        td = 0.0 + gamma * boot - float(Phi[s] @ w)
        w = w + lr * td * Phi[s]
        if use_target and (t + 1) % sync == 0:
            wt = w.copy()
        norms[t] = float(np.linalg.norm(w))
    return norms

norm_plain = run(False)
norm_target = run(True)

answers = {
    "norm_plain": norm_plain,
    "norm_target": norm_target,
    "final_plain": float(norm_plain[-1]),
    "final_target": float(norm_target[-1]),
    "growth_plain": float(norm_plain[-1] / np.linalg.norm(w0)),
}
```